
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Deploying an Observable Agent

## Overview

This demo progressively builds a production-ready agent with deployment, observability, and tool integration in a single flow. You will start with a bare-bones agent deployed as a Databricks App via Declarative Automation Bundles (DABs), layer on MLflow tracing for full request-level visibility, and then attach a Unity Catalog function as a tool through the Model Context Protocol (MCP). By the end, every agent interaction (including tool calls) will produce structured MLflow traces you can inspect in the Databricks UI.

## Learning Objectives

By the end of this demonstration, you will be able to:

1. Deploy an agent on Databricks Apps using DABs
2. Add MLflow tracing for observability
3. Integrate tools via MCP and Unity Catalog functions
4. Verify traces in the MLflow UI

## A. Required Compute Environment

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Serverless Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

<div style="
border-left: 4px solid #F44336;
background: #FFEBEE;
padding: 14px 18px;
border-radius: 4px;
margin: 16px 0;
">
<strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">GIT FOLDER REQUIREMENT</strong>
<div style="color:#333;">

The Declarative Automation Bundles (DABs) UI is only available when you are working inside a Git folder in the workspace.

Before opening the DABs UI, make sure you have:
- Created or cloned a Git folder in the workspace with all associated labs files
- Git is outside the scope of this lab. For more information visit the Databricks Git folders ([AWS](https://docs.databricks.com/aws/en/repos/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/repos/) | [GCP](https://docs.databricks.com/gcp/en/repos/)) documentation.
- **If you are working in Vocareum:** Vocareum does not automatically come with a git folder as a part of the workspace creation. Please use the official documentation to set up Git Folders if working in **Databricks Workspace**.
</div>
</div>

## B. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-2

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 38.1 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4f11dad0-d016-4994-909b-12b7b34410af
    Can't uninstall 'PyYAML'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which versio

Starting Demo Environment Setup
Using config: Includes/config/config_2.yaml

[0] Compute check passed: Serverless 5 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17710504_1793019265' found. Using it.

[2] Setting up schema...
  Catalog : labuser17710504_1793019265
  Schema  : agent_deployment

[3] Creating folder...
Folder created: ./simple-agent

[4] Creating files...
File created: ./simple-agent/agent.py
File created: ./simple-agent/app.yaml
File created: ./simple-agent/databricks.yml
File created: ./simple-agent/pyproject.toml
File created: ./simple-agent/requirements.txt
File created: ./simple-agent/server.py
File created: ./simple-agent/chat.html

[5] Processing datasets...
  Using Vocareum Delta Share: airbnb_sample_data

[6] Registering 1 SQL UC tool(s)...

Creating Tools
  UC function created: avg_neigh_price (from sql/)
Refreshing environment

Demo Environment Setup Complete


Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## C. Define and Deploy a Basic Agent

In this section you will create the files needed to deploy a simple chatbot as a Databricks App. The agent uses the OpenAI Agents SDK with its client redirected to a Databricks AI Model Serving endpoint.

**Note🚨:** Please replace the code below in the files inside the **`simple-agent`** folder.

### C1. Project Files

The basic agent requires six files. Select each tab below to view the file contents and an explanation of its role.

<style>
.c1-tab-row { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 0; }
.c1-tab {
  background: #F9F7F4; border: 2px solid transparent; border-top: 5px solid #1B5162;
  border-radius: 8px 8px 0 0; padding: 10px 18px; cursor: pointer; user-select: none;
  font-weight: 700; font-size: 13pt; color: #0b2026; transition: background 0.15s, transform 0.1s;
}
.c1-tab:hover { transform: translateY(-2px); }
.c1-tab.c1-active { background: #fff; border-left-color: #1B5162; border-right-color: #1B5162; border-bottom-color: #fff; }
.c1-panel-wrap {
  border: 2px solid #1B5162; border-top: none; border-radius: 0 0 10px 10px;
  background: #fff; overflow: hidden; max-height: 0; opacity: 0;
  transition: max-height 0.4s ease, opacity 0.3s ease;
}
.c1-panel-wrap.c1-open { max-height: 4000px; opacity: 1; }
.c1-panel { padding: 20px 24px; }
.c1-panel p { font-size: 14pt; line-height: 1.6; color: #333; margin: 0 0 12px 0; }
.c1-panel ul { margin: 8px 0 12px 20px; font-size: 14pt; line-height: 1.6; color: #333; }
.c1-panel li { margin-bottom: 6px; }
.c1-code-wrap { position: relative; margin: 12px 0 16px 0; }
.c1-copy-btn {
  position: absolute; top: 10px; right: 12px; z-index: 2; padding: 5px 12px;
  font-size: 11pt; font-weight: 600; background: rgba(0,0,0,0.06); color: #383a42;
  border: 1px solid rgba(0,0,0,0.1); border-radius: 6px; cursor: pointer; transition: background 0.15s, color 0.15s, border-color 0.15s;
}
.c1-copy-btn:hover { background: rgba(0,0,0,0.1); }
.c1-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; padding-top: 44px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
}
.c1-code .tk-kw { color: #a626a4; } .c1-code .tk-fn { color: #4078f2; }
.c1-code .tk-str { color: #50a14f; } .c1-code .tk-num { color: #986801; }
.c1-code .tk-cmt { color: #a0a1a7; } .c1-code .tk-dim { color: #696c77; }
.c1-code .tk-key { color: #e45649; }
.c1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, color 0.25s; }
</style>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">

<div class="c1-tab-row">
  <div class="c1-tab c1-active" data-c1tab="0" onclick="c1TabSelect(0)">agent.py</div>
  <div class="c1-tab" data-c1tab="1" onclick="c1TabSelect(1)">server.py</div>
  <div class="c1-tab" data-c1tab="2" onclick="c1TabSelect(2)">pyproject.toml</div>
  <div class="c1-tab" data-c1tab="3" onclick="c1TabSelect(3)">app.yaml</div>
  <div class="c1-tab" data-c1tab="4" onclick="c1TabSelect(4)">databricks.yml</div>
  <div class="c1-tab" data-c1tab="5" onclick="c1TabSelect(5)">requirements.txt</div>
</div>

<div class="c1-panel-wrap c1-open" id="c1-panel-wrap">
  <div class="c1-panel" id="c1-panel"></div>
</div>

</div>

<script>
var C1_FILES = [
  {
    name: "agent.py",
    lang: "python",
    desc: '<p>This file defines the agent and an async function to run it. The key integration point is <code>set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))</code>, which routes all model calls through Databricks instead of the public OpenAI service.</p><p><strong>Key components:</strong></p><ul><li><code>Agent</code> and <code>Runner</code>: Define and execute the agent workflow from the OpenAI Agents SDK</li><li><code>AsyncDatabricksOpenAI</code>: An async client that communicates with Databricks AI Model Serving endpoints using the OpenAI-compatible interface</li><li><code>set_default_openai_api("chat_completions")</code>: Ensures requests use the chat completions format</li><li><code>SERVING_ENDPOINT_NAME</code>: Read from an environment variable injected by the DAB at deploy time</li></ul>',
    code: 'from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport os\n\n# Read the endpoint name injected by Databricks Apps\nSERVING_ENDPOINT_NAME = os.getenv("SERVING_ENDPOINT_NAME")\n\n# Point the OpenAI Agents SDK at Databricks instead of OpenAI\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api("chat_completions")\n\n# Define the agent\nagent = Agent(\n    name="Simple Chatbot",\n    instructions="You are a helpful assistant. Answer the user\'s questions clearly and concisely.",\n    model=SERVING_ENDPOINT_NAME,\n)\n\n\nasync def run_agent(messages: list[dict]) -> str:\n    """Run the agent with a list of messages and return the final reply."""\n    result = await Runner.run(agent, messages)\n    return result.final_output'
  },
  {
    name: "server.py",
    lang: "python",
    desc: '<p>This FastAPI server exposes two endpoints: <code>GET /</code> returns a chat HTML page, and <code>POST /invocations</code> forwards conversation messages to the agent and returns the reply.</p>',
    code: 'import uvicorn\nfrom pathlib import Path\n\nfrom fastapi import FastAPI\nfrom fastapi.responses import HTMLResponse\nfrom pydantic import BaseModel\n\nfrom agent import run_agent\n\napp = FastAPI()\n\nCHAT_UI = Path(__file__).parent / "chat.html"\n\n\nclass ChatRequest(BaseModel):\n    input: list[dict]\n\n\n@app.get("/", response_class=HTMLResponse)\nasync def root():\n    return CHAT_UI.read_text()\n\n\n@app.post("/invocations")\nasync def invocations(request: ChatRequest):\n    reply = await run_agent(request.input)\n    return {"output": reply}\n\n\ndef main():\n    uvicorn.run("server:app", host="0.0.0.0", port=8000)\n\n\nif __name__ == "__main__":\n    main()'
  },
  {
    name: "pyproject.toml",
    lang: "toml",
    desc: '<p>Standard Python project metadata and dependency declaration. The <code>[project.scripts]</code> entry creates the <code>start-server</code> CLI command referenced by <code>app.yaml</code>.</p>',
    code: '[project]\nname = "simple-chatbot"\nversion = "0.1.0"\nrequires-python = ">=3.11"\ndependencies = [\n    "openai-agents>=0.6.0",\n    "mcp<2",\n   "databricks-openai==0.15.0",\n    "databricks-sdk==0.108.0",\n    "fastapi>=0.115.0",\n    "uvicorn>=0.34.0",\n    "python-dotenv",\n    "uv",\n    "databricks-vectorsearch==0.68",\n]\n\n[build-system]\nrequires = ["hatchling"]\nbuild-backend = "hatchling.build"\n\n[tool.hatch.build.targets.wheel]\ninclude = ["agent.py", "server.py"]\n\n[project.scripts]\nstart-server = "server:main"'
  },
  {
    name: "app.yaml",
    lang: "yaml",
    desc: '<p>The Databricks App manifest. It declares which environment variables to inject (resolved from DAB resources) and the startup command.</p>',
    code: 'env:\n  - name: SERVING_ENDPOINT_NAME\n    valueFrom: serving-endpoint\n\ncommand: ["uv", "run", "start-server"]'
  },
  {
    name: "databricks.yml",
    lang: "yaml",
    desc: '<p>The Declarative Automation Bundle (DAB) manifest. It names the app, grants it access to a model serving endpoint, and targets the <code>dev</code> deployment mode. The app name uses bundle substitutions (<code>${workspace.current_user.id}</code>, <code>${bundle.target}</code>) so each user gets a unique app.</p><div class="callout" style="border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;"><strong>Not in a Git folder?</strong> This file is only needed for DABs-based deployment. If you are deploying via the Apps UI instead, skip this file and configure the app resources manually on the app Overview page.</div>',
    code: 'bundle:\n  name: simple_chatbot\n\nresources:\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: "Simple chatbot agent"\n      source_code_path: ./\n      resources:\n        - name: serving-endpoint\n          serving_endpoint:\n            name: "databricks-gpt-5-2"\n            permission: CAN_QUERY\n\ntargets:\n  dev:\n    mode: development\n    default: true'
  },
  {
    name: "requirements.txt",
    lang: "text",
    desc: '<p>Databricks Apps automatically runs <code>pip install -r requirements.txt</code> during deployment. Installing <code>uv</code> here ensures the <code>uv run start-server</code> command in <code>app.yaml</code> works correctly.</p>',
    code: 'uv'
  }
];

var c1TabCurrent = 0;

function c1Esc(s) { return s.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;'); }

function c1PyTokenize(line) {
  var out = '';
  var re = /(#.*$)|("""[\s\S]*?"""|'''[\s\S]*?'''|"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:from|import|def|class|return|if|elif|else|for|while|try|except|finally|with|as|pass|break|continue|yield|lambda|async|await|None|True|False|and|or|not|in|is)\b)|(@\w[\w.]*)|(\b\w+(?=\s*\())|(\b\d+\.?\d*\b)/g;
  var last = 0; var m;
  while ((m = re.exec(line)) !== null) {
    if (m.index > last) out += c1Esc(line.substring(last, m.index));
    if (m[1]) out += '<span class="tk-cmt">' + c1Esc(m[1]) + '</span>';
    else if (m[2]) out += '<span class="tk-str">' + c1Esc(m[2]) + '</span>';
    else if (m[3]) out += '<span class="tk-kw">' + c1Esc(m[3]) + '</span>';
    else if (m[4]) out += '<span class="tk-fn">' + c1Esc(m[4]) + '</span>';
    else if (m[5]) out += '<span class="tk-fn">' + c1Esc(m[5]) + '</span>';
    else if (m[6]) out += '<span class="tk-num">' + c1Esc(m[6]) + '</span>';
    last = m.index + m[0].length;
  }
  if (last < line.length) out += c1Esc(line.substring(last));
  return out || '\u00a0';
}

function c1YamlTokenize(line) {
  var out = '';
  var re = /(#.*$)|("(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:true|false|null)\b)|(^\s*)([\w][\w\-\.\[\]]*)(\s*[:=])/gm;
  var last = 0; var m;
  while ((m = re.exec(line)) !== null) {
    if (m.index > last) out += c1Esc(line.substring(last, m.index));
    if (m[1]) out += '<span class="tk-cmt">' + c1Esc(m[1]) + '</span>';
    else if (m[2]) out += '<span class="tk-str">' + c1Esc(m[2]) + '</span>';
    else if (m[3]) out += '<span class="tk-kw">' + c1Esc(m[3]) + '</span>';
    else if (m[5]) { out += c1Esc(m[4]) + '<span class="tk-key">' + c1Esc(m[5]) + '</span>' + c1Esc(m[6]); }
    last = m.index + m[0].length;
  }
  if (last < line.length) out += c1Esc(line.substring(last));
  return out || '\u00a0';
}

function c1Tokenize(code, lang) {
  var lines = code.split('\n');
  var result = [];
  for (var i = 0; i < lines.length; i++) {
    var raw = lines[i];
    var indent = 0;
    while (indent < raw.length && raw[indent] === ' ') indent++;
    var sp = '';
    for (var j = 0; j < indent; j++) sp += '\u00a0';
    var content = raw.substring(indent);
    var html;
    if (lang === 'python') html = c1PyTokenize(content);
    else if (lang === 'yaml' || lang === 'toml') html = c1YamlTokenize(content);
    else html = c1Esc(content) || '\u00a0';
    result.push('<span class="line">' + sp + html + '</span>');
  }
  return result.join('');
}

function c1TabSelect(id) {
  c1TabCurrent = id;
  document.querySelectorAll('.c1-tab').forEach(function(t) {
    t.classList.toggle('c1-active', parseInt(t.getAttribute('data-c1tab')) === id);
  });
  var f = C1_FILES[id];
  var btnId = 'c1-copy-btn-' + id;
  var wrapId = 'c1-code-wrap';
  document.getElementById('c1-panel').innerHTML =
    f.desc +
    '<div class="c1-code-wrap" id="' + wrapId + '">' +
      '<button class="c1-copy-btn" id="' + btnId + '" onclick="c1CopyCode(' + id + ')">Copy</button>' +
      '<div class="c1-code">' + c1Tokenize(f.code, f.lang) + '</div>' +
    '</div>';
  document.getElementById('c1-panel-wrap').classList.add('c1-open');
}

function c1CopyCode(id) {
  var code = C1_FILES[id].code;
  var btn = document.getElementById('c1-copy-btn-' + id);
  navigator.clipboard.writeText(code).then(function() {
    btn.textContent = 'Copied!';
    setTimeout(function() { btn.textContent = 'Copy'; }, 2000);
  }, function() {
    var t = document.createElement('textarea');
    t.value = code; document.body.appendChild(t); t.select();
    document.execCommand('copy'); document.body.removeChild(t);
    btn.textContent = 'Copied!';
    setTimeout(function() { btn.textContent = 'Copy'; }, 2000);
  });
}

c1TabSelect(0);
</script>

Run the next cell to generate a unique identifier if you are deploying the app via the UI. That is, you are _not_ working in a git folder.

In [0]:
user_email = spark.sql("SELECT current_user()").first()[0]
uniqueIdentifier1 = user_email.split("@")[0].replace('labuser','').replace("_","-").replace(".","-")
print("Use the following app name if you are deploying your app via the UI:\n")
print(f"agent-{uniqueIdentifier1}-dev")

Use the following app name if you are deploying your app via the UI:

agent-17710504-1793019265-dev


### C2. Deploy the App

There are two ways to deploy, depending on whether your workspace files are inside a Git folder. Select a card below.

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">

<style>
.c2d-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }

.c2d-box {
  display: inline-block;
  width: 48%;
  margin-right: 4%;
  min-height: 90px;
  background: #F9F7F4;
  border-top: 8px solid transparent;
  border-left: 2px solid transparent;
  border-right: 2px solid transparent;
  border-bottom: 2px solid transparent;
  border-radius: 8px;
  padding: 12px 8px;
  text-align: center;
  box-sizing: border-box;
  cursor: pointer;
  user-select: none;
  vertical-align: top;
  transition: transform 0.12s;
}

.c2d-box:last-child { margin-right: 0; }

.c2d-box:hover { transform: translateY(-2px); }

.c2d-box.active {
  background: #fff;
  border-left-color: var(--pc);
  border-right-color: var(--pc);
  border-bottom-color: var(--pc);
}

.c2d-title {
  display: block;
  font-size: 13pt;
  font-weight: 700;
  color: #0b2026;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
}

.c2d-subtitle {
  display: block;
  font-size: 11pt;
  font-weight: 400;
  color: #666;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
  margin-top: 4px;
}

.c2d-detail-wrap {
  overflow: hidden;
  max-height: 0;
  opacity: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
  margin-top: 0;
}

.c2d-detail-wrap.open {
  max-height: 1600px;
  opacity: 1;
  margin-top: 12px;
}

.c2d-detail-card {
  background: #F9F7F4;
  border-radius: 10px;
  padding: 20px;
  box-sizing: border-box;
  border-top: 7px solid #ccc;
}

.c2d-steps-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 14px 14px;
  margin-bottom: 14px;
  font-size: 14.5pt;
  line-height: 1.6;
}

.c2d-steps-block ol {
  margin: 8px 0 0 0;
  padding-left: 20px;
}

.c2d-steps-block li {
  margin-bottom: 8px;
}

.c2d-steps-block ul {
  margin: 4px 0 8px 20px;
  list-style: disc;
}

.c2d-info-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 12px 14px;
  font-size: 14pt;
  line-height: 1.5;
  font-weight: 500;
}
</style>

<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box;">

  <div class="c2d-row">
    <span class="c2d-box active" data-id="0" onclick="c2dSelect(0)" style="--pc:#1B5162; border-top-color:#1B5162;">
      <span class="c2d-title">DABs UI</span>
      <span class="c2d-subtitle">Git folder</span>
    </span><span class="c2d-box" data-id="1" onclick="c2dSelect(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
      <span class="c2d-title">Apps UI</span>
      <span class="c2d-subtitle">No Git folder</span>
    </span>
  </div>

  <div class="c2d-detail-wrap" id="c2d-detail-wrap">
    <div class="c2d-detail-card" id="c2d-detail-card"></div>
  </div>

</div>

</div>

<script>
var C2D_CARDS = [
  {
    title: "DABs UI (Git folder)",
    color: "#1B5162",
    bg: "rgba(27,81,98,0.10)",
    infoBg: "rgba(27,81,98,0.10)",
    content: `
      <ol>
        <li>Navigate inside the project folder and click on any file (e.g., <strong>databricks.yml</strong>). A rocket icon will appear. Click it.</li>
        <li>Click <strong>Deploy</strong> in the <strong>Deployments</strong> section. This DAB is configured to deploy to <strong>dev</strong> only.</li>
        <li>Once validation clears and the deployment summary appears, click <strong>Deploy</strong> again to confirm.</li>
        <li>Under <strong>Bundle resource</strong>, click the app link (e.g., <strong>agent-userid-dev</strong>) to open the <strong>Apps</strong> page.</li>
        <li>Click <strong>Start</strong> to provision compute. After a few minutes, click <strong>Deploy</strong> to push the source code to the running app.
          <ul>
            <li>Navigate back to the source folder of the app <strong>simple-agent</strong> and click the three vertical dots next to the folder name. Select <strong>Copy URL/path</strong> and select <strong>full path</strong>.</li>
            <li>Paste this in the <strong>source code path</strong> textbox in the app's <strong>Create deployment</strong> menu.</li>
            <li>Click <strong>Deploy</strong>.</li>
          </ul>
        </li>
        <li>Once the app shows <strong>Running</strong> status, click the endpoint to open the chat UI and verify the agent responds by sending a query like <em>"Please tell me about Apache Spark."</em></li>
      </ol>
    `,
    info: '<strong>Requires a Git folder.</strong> The DABs UI automates resource creation and permission grants from your <code>databricks.yml</code> file.'
  },
  {
    title: "Apps UI (no Git folder)",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    infoBg: "rgba(255,95,70,0.10)",
    content: `
      <ol>
        <li>In the workspace sidebar, click <strong>+ New</strong> and select <strong>App</strong>.</li>
        <li>Click <strong>Create a custom app</strong> in the <strong>Apps</strong> landing page.</li>
        <li>Give the app a name (e.g., <strong>agent-uniqueIdentifier-dev</strong> as indicated in the previous cell) and click <strong>Create app</strong>.</li>
        <li>Wait for the compute to provision.
        </li>
        <li>Configure the app resources manually: add a <strong>model service</strong> resource and map it to <code>GPT 5.2</code> with <code>CAN EXECUTE</code> permission.
          <ul>
            <li>Click on <strong>(⋮)</strong> icon then on  <strong>Edit</strong> at the top right.</li>
            <li>Click <strong>Resources</strong> at the top of the setup wizard.</li>
            <li>Click <strong>+ Add resource</strong> and select <strong>Model</strong>.</li>
            <li>In the dropdown menu search for <strong>GPT 5.2</strong>. Make sure <strong>Permission</strong> is set to <strong>Can execute</strong>. Make sure <strong>Resource key</strong> is set to <strong>serving-endpoint</strong>.</li>
            <li>Click <strong>Save</strong>.</li>
          </ul>
        </li>
        <li>On the app's Overview page, click <strong>Deploy</strong> and set the <strong>Source path</strong> to the workspace folder containing your project files.
          <ul>
            <li>Navigate back to the source folder of the app <strong>simple-agent</strong> and click the three vertical dots next to the folder name. Select <strong>Copy URL/path</strong> and select <strong>full path</strong>.</li>
            <li>Paste this in the <strong>source code path</strong> textbox in the app's <strong>Create deployment</strong> menu.</li>
            <li>Click <strong>Deploy</strong>.</li>
          </ul>
        </li>
        <li>Click the endpoint URL to open the chat UI and verify the agent responds by sending a query like <em>"Please tell me about Apache Spark."</em></li>
      </ol>
    `,
    info: '<strong>No Git folder required.</strong> You configure app resources manually on the Overview page.'
  }
];

var c2dCurrent = null;

function c2dSelect(id) {
  var wrap = document.getElementById('c2d-detail-wrap');
  var card = document.getElementById('c2d-detail-card');
  var c = C2D_CARDS[id];

  document.querySelectorAll('.c2d-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (c2dCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.c2d-box').forEach(function(b) {
      b.classList.remove('active');
    });
    c2dCurrent = null;
    return;
  }

  c2dCurrent = id;

  card.style.borderTopColor = c.color;
  card.innerHTML =
    '<div style="font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;">' + c.title + '</div>'
    + '<div class="c2d-steps-block" style="background:' + c.bg + ';border-color:' + c.color + ';">'
    + c.content
    + '</div>'
    + '<div class="c2d-info-block" style="background:' + c.infoBg + ';border-color:' + c.color + ';">'
    + c.info
    + '</div>';

  wrap.classList.add('open');
}

c2dSelect(0);
</script>

## D. Add MLflow Tracing

With the basic agent running, the next step is to add observability. You will update three files to wire in MLflow tracing and experiment tracking so every conversation produces a structured trace.

### D1. Updated Files

Select each tab below to view the updated file contents. Each file now includes MLflow tracing and experiment configuration.

<style>
.d1-tab-row { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 0; }
.d1-tab {
  background: #F9F7F4; border: 2px solid transparent; border-top: 5px solid #2574B5;
  border-radius: 8px 8px 0 0; padding: 10px 18px; cursor: pointer; user-select: none;
  font-weight: 700; font-size: 13pt; color: #0b2026; transition: background 0.15s, transform 0.1s;
}
.d1-tab:hover { transform: translateY(-2px); }
.d1-tab.d1-active { background: #fff; border-left-color: #2574B5; border-right-color: #2574B5; border-bottom-color: #fff; }
.d1-panel-wrap {
  border: 2px solid #2574B5; border-top: none; border-radius: 0 0 10px 10px;
  background: #fff; overflow: hidden; max-height: 0; opacity: 0;
  transition: max-height 0.4s ease, opacity 0.3s ease;
}
.d1-panel-wrap.d1-open { max-height: 4000px; opacity: 1; }
.d1-panel { padding: 20px 24px; }
.d1-panel p { font-size: 14pt; line-height: 1.6; color: #333; margin: 0 0 12px 0; }
.d1-panel ul { margin: 8px 0 12px 20px; font-size: 14pt; line-height: 1.6; color: #333; }
.d1-panel li { margin-bottom: 6px; }
.d1-changed-box {
  border-left: 4px solid #2574B5; background: rgba(37,116,181,0.08);
  border-radius: 0 6px 6px 0; padding: 14px 16px; margin: 14px 0; font-size: 13.5pt; line-height: 1.6;
}
.d1-code-wrap { position: relative; margin: 12px 0 16px 0; }
.d1-copy-btn {
  position: absolute; top: 10px; right: 12px; z-index: 2; padding: 5px 12px;
  font-size: 11pt; font-weight: 600; background: rgba(0,0,0,0.06); color: #383a42;
  border: 1px solid rgba(0,0,0,0.1); border-radius: 6px; cursor: pointer; transition: background 0.15s, color 0.15s, border-color 0.15s;
}
.d1-copy-btn:hover { background: rgba(0,0,0,0.1); }
.d1-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; padding-top: 44px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
}
.d1-code .tk-kw { color: #a626a4; } .d1-code .tk-fn { color: #4078f2; }
.d1-code .tk-str { color: #50a14f; } .d1-code .tk-num { color: #986801; }
.d1-code .tk-cmt { color: #a0a1a7; } .d1-code .tk-dim { color: #696c77; }
.d1-code .tk-key { color: #e45649; }
.d1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, color 0.25s; }
</style>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">

<div class="d1-tab-row">
  <div class="d1-tab d1-active" data-d1tab="0" onclick="d1TabSelect(0)">agent.py</div>
  <div class="d1-tab" data-d1tab="1" onclick="d1TabSelect(1)">databricks.yml</div>
  <div class="d1-tab" data-d1tab="2" onclick="d1TabSelect(2)">app.yaml</div>
</div>

<div class="d1-panel-wrap d1-open" id="d1-panel-wrap">
  <div class="d1-panel" id="d1-panel"></div>
</div>

</div>

<script>
var D1_FILES = [
  {
    name: "agent.py",
    lang: "python",
    desc: '<p>Replace the contents of <strong>agent.py</strong> with the version below. The additions are: <code>mlflow</code> and <code>json</code> imports, tracking URI configuration, experiment setup from environment variables, and the <code>@mlflow.trace</code> decorator with <code>mlflow.start_run</code> logging inside <code>run_agent()</code>.</p>',
    changes: '<strong>What changed:</strong><ul><li><code>mlflow.set_tracking_uri("databricks")</code>: Routes all traces and runs to your Databricks workspace</li><li><code>mlflow.set_experiment()</code>: Targets the experiment created by the DAB, using the ID injected via environment variable</li><li><code>@mlflow.trace</code>: Wraps every call in a root span that captures inputs, outputs, and timing</li><li><code>mlflow.start_run(nested=True)</code>: Logs parameters and artifacts to the experiment for each invocation</li></ul>',
    code: 'from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport json\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api("chat_completions")\n\nmlflow.set_tracking_uri("databricks")\n\n# Read environment variables injected by the DAB\nSERVING_ENDPOINT_NAME = os.getenv("SERVING_ENDPOINT_NAME")\nexperiment_id = os.getenv("MLFLOW_EXPERIMENT_ID")\nexperiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME")\n\nif experiment_id:\n    mlflow.set_experiment(experiment_id=experiment_id)\nelif experiment_name:\n    mlflow.set_experiment(experiment_name)\n\n# Define the agent\nagent = Agent(\n    name="Simple Chatbot",\n    instructions="You are a helpful assistant. Answer the user\'s questions clearly and concisely.",\n    model=SERVING_ENDPOINT_NAME,\n)\n\n\n@mlflow.trace(name="mlflow_chatbot_run", span_type="AGENT", attributes={"agent_name": "MLflow Chatbot"})\nasync def run_agent(messages: list[dict]) -> str:\n    """\n    Run the agent with a list of messages and return the final reply.\n    - @mlflow.trace creates a trace + root span with inputs/outputs.\n    - mlflow.start_run logs params/artifacts to the experiment.\n    """\n    with mlflow.start_run(nested=True):\n        mlflow.log_param("agent_name", agent.name)\n        mlflow.log_param("model_endpoint", agent.model)\n        mlflow.log_param("num_messages", len(messages))\n        mlflow.log_text(json.dumps(messages, indent=2), "inputs/messages.json")\n        result = await Runner.run(agent, messages)\n        final_output = result.final_output\n        mlflow.log_text(final_output, "outputs/final_output.txt")\n        return final_output'
  },
  {
    name: "databricks.yml",
    lang: "yaml",
    desc: '<p>Replace the contents of <strong>databricks.yml</strong> with the version below. This adds an MLflow experiment as a bundle-managed resource and grants the app permission to write to it.</p><div class="callout" style="border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;"><strong>Not in a Git folder?</strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the experiment resource manually on the app Overview page during the deploy step.</div>',
    changes: '<strong>What changed:</strong><ul><li><code>resources.experiments.chatbot_experiment</code>: Declares an MLflow experiment that DABs creates on deploy. This is a new resource added to the same app.</li><li><code>app-experiment</code> resource entry: Grants the app\'s service principal <code>CAN_EDIT</code> on the experiment so it can create runs and log traces</li><li>The app description updated to reflect the new capability. The bundle name and app name remain the same; redeploying updates the existing app in place.</li></ul>',
    code: 'bundle:\n  name: simple_chatbot\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: "Simple chatbot agent"\n      source_code_path: ./\n      resources:\n        - name: serving-endpoint\n          serving_endpoint:\n            name: databricks-gpt-5-2\n            permission: CAN_QUERY\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n\ntargets:\n  dev:\n    mode: development\n    default: true'
  },
  {
    name: "app.yaml",
    lang: "yaml",
    desc: '<p>Replace the contents of <strong>app.yaml</strong> with the version below. This injects the MLflow experiment ID as an environment variable.</p>',
    changes: '<strong>What changed:</strong><ul><li>The new <code>MLFLOW_EXPERIMENT_ID</code> entry uses <code>valueFrom</code> to resolve the experiment ID from the <code>app-experiment</code> resource declared in <code>databricks.yml</code>. The agent reads this at startup to target the correct experiment.</li></ul>',
    code: 'env:\n  - name: SERVING_ENDPOINT_NAME\n    valueFrom: serving-endpoint\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n\ncommand: ["uv", "run", "start-server"]'
  }
];

var d1TabCurrent = 0;

function d1Esc(s) { return s.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;'); }

function d1PyTokenize(line) {
  var out = '';
  var re = /(#.*$)|("""[\s\S]*?"""|'''[\s\S]*?'''|"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:from|import|def|class|return|if|elif|else|for|while|try|except|finally|with|as|pass|break|continue|yield|lambda|async|await|None|True|False|and|or|not|in|is)\b)|(@\w[\w.]*)|(\b\w+(?=\s*\())|(\b\d+\.?\d*\b)/g;
  var last = 0; var m;
  while ((m = re.exec(line)) !== null) {
    if (m.index > last) out += d1Esc(line.substring(last, m.index));
    if (m[1]) out += '<span class="tk-cmt">' + d1Esc(m[1]) + '</span>';
    else if (m[2]) out += '<span class="tk-str">' + d1Esc(m[2]) + '</span>';
    else if (m[3]) out += '<span class="tk-kw">' + d1Esc(m[3]) + '</span>';
    else if (m[4]) out += '<span class="tk-fn">' + d1Esc(m[4]) + '</span>';
    else if (m[5]) out += '<span class="tk-fn">' + d1Esc(m[5]) + '</span>';
    else if (m[6]) out += '<span class="tk-num">' + d1Esc(m[6]) + '</span>';
    last = m.index + m[0].length;
  }
  if (last < line.length) out += d1Esc(line.substring(last));
  return out || '\u00a0';
}

function d1YamlTokenize(line) {
  var out = '';
  var re = /(#.*$)|("(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:true|false|null)\b)|(^\s*)([\w][\w\-\.\[\]]*)(\s*[:=])/gm;
  var last = 0; var m;
  while ((m = re.exec(line)) !== null) {
    if (m.index > last) out += d1Esc(line.substring(last, m.index));
    if (m[1]) out += '<span class="tk-cmt">' + d1Esc(m[1]) + '</span>';
    else if (m[2]) out += '<span class="tk-str">' + d1Esc(m[2]) + '</span>';
    else if (m[3]) out += '<span class="tk-kw">' + d1Esc(m[3]) + '</span>';
    else if (m[5]) { out += d1Esc(m[4]) + '<span class="tk-key">' + d1Esc(m[5]) + '</span>' + d1Esc(m[6]); }
    last = m.index + m[0].length;
  }
  if (last < line.length) out += d1Esc(line.substring(last));
  return out || '\u00a0';
}

function d1Tokenize(code, lang) {
  var lines = code.split('\n');
  var result = [];
  for (var i = 0; i < lines.length; i++) {
    var raw = lines[i];
    var indent = 0;
    while (indent < raw.length && raw[indent] === ' ') indent++;
    var sp = '';
    for (var j = 0; j < indent; j++) sp += '\u00a0';
    var content = raw.substring(indent);
    var html;
    if (lang === 'python') html = d1PyTokenize(content);
    else if (lang === 'yaml' || lang === 'toml') html = d1YamlTokenize(content);
    else html = d1Esc(content) || '\u00a0';
    result.push('<span class="line">' + sp + html + '</span>');
  }
  return result.join('');
}

function d1TabSelect(id) {
  d1TabCurrent = id;
  document.querySelectorAll('.d1-tab').forEach(function(t) {
    t.classList.toggle('d1-active', parseInt(t.getAttribute('data-d1tab')) === id);
  });
  var f = D1_FILES[id];
  var btnId = 'd1-copy-btn-' + id;
  var wrapId = 'd1-code-wrap';
  var html = f.desc;
  if (f.changes) {
    html += '<div class="d1-changed-box">' + f.changes + '</div>';
  }
  html += '<div class="d1-code-wrap" id="' + wrapId + '">' +
      '<button class="d1-copy-btn" id="' + btnId + '" onclick="d1CopyCode(' + id + ')">Copy</button>' +
      '<div class="d1-code">' + d1Tokenize(f.code, f.lang) + '</div>' +
    '</div>';
  document.getElementById('d1-panel').innerHTML = html;
  document.getElementById('d1-panel-wrap').classList.add('d1-open');
}

function d1CopyCode(id) {
  var code = D1_FILES[id].code;
  var btn = document.getElementById('d1-copy-btn-' + id);
  navigator.clipboard.writeText(code).then(function() {
    btn.textContent = 'Copied!';
    setTimeout(function() { btn.textContent = 'Copy'; }, 2000);
  }, function() {
    var t = document.createElement('textarea');
    t.value = code; document.body.appendChild(t); t.select();
    document.execCommand('copy'); document.body.removeChild(t);
    btn.textContent = 'Copied!';
    setTimeout(function() { btn.textContent = 'Copy'; }, 2000);
  });
}

d1TabSelect(0);
</script>

In [0]:
uniqueIdentifier2 = user_email.split("@")[0].replace('labuser','labuser_').replace(".","_")
print("Use the following MLflow name if you are deploying your app via the UI:\n")
print(f"[dev {uniqueIdentifier2}] simple-chatbot-experiment")

Use the following MLflow name if you are deploying your app via the UI:

[dev labuser_17710504_1793019265] simple-chatbot-experiment


### D2. Redeploy

Redeploy using whichever method you used in Section C2. This update adds an MLflow experiment resource to the app. Select a card below.

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">

<style>
.d2d-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }

.d2d-box {
  display: inline-block;
  width: 48%;
  margin-right: 4%;
  min-height: 90px;
  background: #F9F7F4;
  border-top: 8px solid transparent;
  border-left: 2px solid transparent;
  border-right: 2px solid transparent;
  border-bottom: 2px solid transparent;
  border-radius: 8px;
  padding: 12px 8px;
  text-align: center;
  box-sizing: border-box;
  cursor: pointer;
  user-select: none;
  vertical-align: top;
  transition: transform 0.12s;
}

.d2d-box:last-child { margin-right: 0; }

.d2d-box:hover { transform: translateY(-2px); }

.d2d-box.active {
  background: #fff;
  border-left-color: var(--pc);
  border-right-color: var(--pc);
  border-bottom-color: var(--pc);
}

.d2d-title {
  display: block;
  font-size: 13pt;
  font-weight: 700;
  color: #0b2026;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
}

.d2d-subtitle {
  display: block;
  font-size: 11pt;
  font-weight: 400;
  color: #666;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
  margin-top: 4px;
}

.d2d-detail-wrap {
  overflow: hidden;
  max-height: 0;
  opacity: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
  margin-top: 0;
}

.d2d-detail-wrap.open {
  max-height: 1600px;
  opacity: 1;
  margin-top: 12px;
}

.d2d-detail-card {
  background: #F9F7F4;
  border-radius: 10px;
  padding: 20px;
  box-sizing: border-box;
  border-top: 7px solid #ccc;
}

.d2d-steps-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 14px 14px;
  margin-bottom: 14px;
  font-size: 14.5pt;
  line-height: 1.6;
}

.d2d-steps-block ol {
  margin: 8px 0 0 0;
  padding-left: 20px;
}

.d2d-steps-block li {
  margin-bottom: 8px;
}

.d2d-steps-block ul {
  margin: 4px 0 8px 20px;
  list-style: disc;
}

.d2d-info-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 12px 14px;
  font-size: 14pt;
  line-height: 1.5;
  font-weight: 500;
}
</style>

<div style="background: #F8F9FC; border: 3px solid #2574B5; border-radius: 10px; padding: 24px; box-sizing: border-box;">

  <div class="d2d-row">
    <span class="d2d-box active" data-id="0" onclick="d2dSelect(0)" style="--pc:#2574B5; border-top-color:#2574B5;">
      <span class="d2d-title">DABs UI</span>
      <span class="d2d-subtitle">Git folder</span>
    </span><span class="d2d-box" data-id="1" onclick="d2dSelect(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
      <span class="d2d-title">Apps UI</span>
      <span class="d2d-subtitle">No Git folder</span>
    </span>
  </div>

  <div class="d2d-detail-wrap" id="d2d-detail-wrap">
    <div class="d2d-detail-card" id="d2d-detail-card"></div>
  </div>

</div>

</div>

<script>
var D2D_CARDS = [
  {
    title: "DABs UI (Git folder)",
    color: "#2574B5",
    bg: "rgba(37,116,181,0.10)",
    infoBg: "rgba(37,116,181,0.10)",
    content: `
      <ol>
        <li>Click the rocket icon on any file in the project folder.</li>
        <li>Click <strong>Deploy</strong>, wait for validation, then click <strong>Deploy</strong> again.</li>
        <li>Navigate to the app and click <strong>Deploy</strong> to push the updated source.</li>
        <li>Notice that you now have a new <strong>App resource</strong> for your <strong>MLflow experiment</strong>.</li>
        <li>After deployment, click on the URL to open the app.</li>
      </ol>
    `,
    info: '<strong>Requires a Git folder.</strong> The DABs UI detects the new experiment resource in <code>databricks.yml</code> and creates it automatically. Wait for the app status to return to <strong>Running</strong>.'
  },
  {
    title: "Apps UI (no Git folder)",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    infoBg: "rgba(255,95,70,0.10)",
    content: `
      <ol>
        <li>Navigate to the app Overview page and click <strong>(⋮)</strong> icon then on  <strong>Edit</strong> at the top right.</li>
        <li>Click <strong>resources</strong> at the top of the setup wizard.</li>
        <li>Click <strong>+ Add resource</strong> and select <strong>MLflow experiment</strong>.</li>
        <li>In the <strong>Select experiment</strong> dropdown, select <strong>Create new experiment</strong>.</li>
        <li>Give it the name <strong>[dev labuser_XXX] simple-chatbot-experiment</strong>, using your own labuser name as printed above.</li>
        <li>For the <strong>trace storage location</strong> select <strong>In the Experiment</strong>. Note that there is an option for storing MLflow experiments in <strong>Unity Catalog</strong>, which we will not cover in this course. 
        <li>Click <strong>Create experiment</strong>.</li>
        <li>Make sure <strong>Permission</strong> is set to <strong>Can edit</strong> and <strong>Resource key</strong> is set to <strong>app-experiment</strong>.</li>
        <li>Click <strong>Save</strong>.</li>
        <li>Back on the Overview page, click <strong>Deploy</strong> to push the updated source.</li>
      </ol>
    `,
    info: '<strong>No Git folder required.</strong> You add the experiment resource manually on the Overview page. Wait for the app status to return to <strong>Running</strong>.'
  }
];

var d2dCurrent = null;

function d2dSelect(id) {
  var wrap = document.getElementById('d2d-detail-wrap');
  var card = document.getElementById('d2d-detail-card');
  var c = D2D_CARDS[id];

  document.querySelectorAll('.d2d-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (d2dCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.d2d-box').forEach(function(b) {
      b.classList.remove('active');
    });
    d2dCurrent = null;
    return;
  }

  d2dCurrent = id;

  card.style.borderTopColor = c.color;
  card.innerHTML =
    '<div style="font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;">' + c.title + '</div>'
    + '<div class="d2d-steps-block" style="background:' + c.bg + ';border-color:' + c.color + ';">'
    + c.content
    + '</div>'
    + '<div class="d2d-info-block" style="background:' + c.infoBg + ';border-color:' + c.color + ';">'
    + c.info
    + '</div>';

  wrap.classList.add('open');
}

d2dSelect(0);
</script>

### D3. Test and Verify Traces

With MLflow tracing wired in, every conversation now produces a structured trace. Verify this is working before moving on.

1. Open the app URL and send the same query as last time: "Please tell me about Apache Spark."
2. Wait for the agent to respond.
3. Navigate to the **Apps** page in the workspace sidebar and select your app.
4. In the **App resources** section, locate the **MLflow Experiment ID**. Click on it to open the experiment.
5. Click the **Traces** tab. You should see a trace entry for the query you just sent.
6. Click the traces to expand it. You will see:
   - A **root span** (`mlflow_chatbot_run` is located under **Details & Timeline**) capturing the full agent invocation with inputs, output, and total latency
   - **Child spans** from the Agents SDK runner showing each step the agent took will appear here as well (though this agent is too simple to have any at this point)

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">At this stage, the trace shows the LLM interaction but no tool calls. The agent can only chat. In the next section, you will add MCP tools and see tool call spans appear as children within the trace.</p>
</div>
</div>
</div>

## E. Add MCP Tools

With tracing in place, you will now extend the agent to call external tools via the Model Context Protocol (MCP). MCP is an open standard that lets AI applications discover and invoke tools hosted on external servers. On Databricks, Managed MCP servers can expose Unity Catalog functions, Genie spaces, and AI Search indexes through a single interface.

### E1. Final Files

Select each tab below to view the final file versions with MCP tool integration. Each tab highlights what changed from Section D.

<style>
.e1-tab-row { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 0; }
.e1-tab {
  background: #F9F7F4; border: 2px solid transparent; border-top: 5px solid #02A36F;
  border-radius: 8px 8px 0 0; padding: 10px 18px; cursor: pointer; user-select: none;
  font-weight: 700; font-size: 13pt; color: #0b2026; transition: background 0.15s, transform 0.1s;
}
.e1-tab:hover { transform: translateY(-2px); }
.e1-tab.e1-active { background: #fff; border-left-color: #02A36F; border-right-color: #02A36F; border-bottom-color: #fff; }
.e1-panel-wrap {
  border: 2px solid #02A36F; border-top: none; border-radius: 0 0 10px 10px;
  background: #fff; overflow: hidden; max-height: 0; opacity: 0;
  transition: max-height 0.4s ease, opacity 0.3s ease;
}
.e1-panel-wrap.e1-open { max-height: 4000px; opacity: 1; }
.e1-panel { padding: 20px 24px; }
.e1-panel p { font-size: 14pt; line-height: 1.6; color: #333; margin: 0 0 12px 0; }
.e1-panel ul { margin: 8px 0 12px 20px; font-size: 14pt; line-height: 1.6; color: #333; }
.e1-panel li { margin-bottom: 6px; }
.e1-changed-box {
  border-left: 4px solid #02A36F; background: rgba(2,163,111,0.08);
  border-radius: 0 6px 6px 0; padding: 14px 16px; margin: 14px 0; font-size: 13.5pt; line-height: 1.6;
}
.e1-code-wrap { position: relative; margin: 12px 0 16px 0; }
.e1-copy-btn {
  position: absolute; top: 10px; right: 12px; z-index: 2; padding: 5px 12px;
  font-size: 11pt; font-weight: 600; background: rgba(0,0,0,0.06); color: #383a42;
  border: 1px solid rgba(0,0,0,0.1); border-radius: 6px; cursor: pointer; transition: background 0.15s, color 0.15s, border-color 0.15s;
}
.e1-copy-btn:hover { background: rgba(0,0,0,0.1); }
.e1-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; padding-top: 44px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
}
.e1-code .tk-kw { color: #a626a4; } .e1-code .tk-fn { color: #4078f2; }
.e1-code .tk-str { color: #50a14f; } .e1-code .tk-num { color: #986801; }
.e1-code .tk-cmt { color: #a0a1a7; } .e1-code .tk-dim { color: #696c77; }
.e1-code .tk-key { color: #e45649; }
.e1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, color 0.25s; }
</style>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">

<div class="e1-tab-row">
  <div class="e1-tab e1-active" data-e1tab="0" onclick="e1TabSelect(0)">agent.py</div>
  <div class="e1-tab" data-e1tab="1" onclick="e1TabSelect(1)">app.yaml</div>
  <div class="e1-tab" data-e1tab="2" onclick="e1TabSelect(2)">databricks.yml</div>
</div>

<div class="e1-panel-wrap e1-open" id="e1-panel-wrap">
  <div class="e1-panel" id="e1-panel"></div>
</div>

</div>

<script>
var E1_FILES = [
  {
    name: "agent.py",
    lang: "python",
    desc: '<p>Replace the contents of <strong>agent.py</strong> with the final version below. This adds MCP imports, a <code>WorkspaceClient</code> for authentication, an <code>McpServer</code> instance pointing at a UC function, the <code>mcp_servers</code> parameter on the Agent, and the <code>async with mcp_server:</code> context manager around <code>Runner.run()</code>.</p>',
    changes: '<strong>What changed from Section D:</strong><ul><li><code>McpServer</code> and <code>WorkspaceClient</code> imports added</li><li><code>McpServer.from_uc_function()</code>: Creates a handle to the Managed MCP server exposing a UC function as a tool</li><li><code>mcp_servers=[mcp_server]</code> on the <code>Agent</code> constructor: Registers the MCP server so the SDK discovers available tools at runtime</li><li><code>async with mcp_server:</code>: Opens the MCP connection before <code>Runner.run()</code> and closes it after, ensuring tools are available during execution</li></ul>',
    code: 'from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom databricks_openai import AsyncDatabricksOpenAI\nfrom databricks_openai.agents import McpServer\nfrom databricks.sdk import WorkspaceClient\nimport json\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api("chat_completions")\n\nmlflow.set_tracking_uri("databricks")\n\n# Read environment variables injected by the DAB\nSERVING_ENDPOINT_NAME = os.getenv("SERVING_ENDPOINT_NAME")\nexperiment_id = os.getenv("MLFLOW_EXPERIMENT_ID")\nexperiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME")\n\nif experiment_id:\n    mlflow.set_experiment(experiment_id=experiment_id)\nelif experiment_name:\n    mlflow.set_experiment(experiment_name)\n\n# MCP configuration\ncatalog_name = os.getenv("CATALOG_NAME")\nfunction_name = os.getenv("FUNCTION_1_NAME")\n\nws = WorkspaceClient()\n\nmcp_server = McpServer.from_uc_function(\n    catalog=catalog_name,\n    schema="agent_deployment",\n    workspace_client=ws,\n    timeout=60.0,\n)\n\n# Define the agent with MCP tool access\nagent = Agent(\n    name="Simple Chatbot",\n    instructions="You are a helpful assistant. Answer the user\'s questions clearly and concisely.",\n    model=SERVING_ENDPOINT_NAME,\n    mcp_servers=[mcp_server],\n)\n\n\n@mlflow.trace(\n    name="tool_calling_chatbot_run",\n    span_type="AGENT",\n    attributes={"agent_name": "Tool-calling Chatbot"},\n)\nasync def run_agent(messages: list[dict]) -> str:\n    with mlflow.start_run(nested=True):\n        mlflow.log_param("agent_name", agent.name)\n        mlflow.log_param("model_endpoint", agent.model)\n        mlflow.log_param("num_messages", len(messages))\n        mlflow.log_text(json.dumps(messages, indent=2), "inputs/messages.json")\n    # McpServer is an async context manager: Opens the connection before the run\n    async with mcp_server:\n        result = await Runner.run(agent, messages)\n    final_output = result.final_output\n    mlflow.log_text(final_output, "outputs/final_output.txt")\n    return final_output'
  },
  {
    name: "app.yaml",
    lang: "yaml",
    desc: '<p>Replace the contents of <strong>app.yaml</strong> with the final version below. This adds <code>CATALOG_NAME</code> and <code>FUNCTION_1_NAME</code> environment variables.</p>',
    changes: '<strong>What changed from Section D:</strong><ul><li>The two new entries use static <code>value</code> fields rather than <code>valueFrom</code>. Update <code>CATALOG_NAME</code> with your actual catalog name (e.g., <code>labuser_sparky_mcspark</code>). The bundle-level <code>variables</code> block in <code>databricks.yml</code> keeps these values parameterized at the bundle level.</li></ul>',
    code: 'env:\n  - name: SERVING_ENDPOINT_NAME\n    valueFrom: serving-endpoint\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n  - name: CATALOG_NAME\n    value: # <---- Update with your catalog name\n  - name: FUNCTION_1_NAME\n    value: avg_neigh_price\n\ncommand: ["uv", "run", "start-server"]'
  },
  {
    name: "databricks.yml",
    lang: "yaml",
    desc: '<p>Replace the contents of <strong>databricks.yml</strong> with the final version below. This adds a <code>variables</code> block for the catalog and function names, and a <code>uc_securable</code> resource entry to grant the app <code>EXECUTE</code> permission on the UC function.</p><div class="callout" style="border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;"><strong>Not in a Git folder?</strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the UC function resource with <code>EXECUTE</code> permission manually on the app Overview page during the deploy step.</div>',
    changes: '<strong>What changed from Section D:</strong><ul><li><code>variables</code> block: Parameterizes the catalog and function names so they can be overridden at deploy time with <code>--var</code></li><li><code>uc-function-1</code> resource entry: Grants the app\'s service principal <code>EXECUTE</code> permission on the Unity Catalog function so the MCP server can invoke it at runtime</li></ul>',
    code: 'bundle:\n  name: simple_chatbot\n\nvariables:\n  catalog_name:\n    description: "Per-user lab catalog, e.g. labuser_matthew_mccoy"\n    default: # <---- Update with your catalog name\n  function_1_name:\n    description: "Function to grab the average neighborhood price"\n    default: avg_neigh_price\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: "Simple chatbot agent"\n      source_code_path: ./\n      resources:\n        - name: serving-endpoint\n          serving_endpoint:\n            name: databricks-gpt-5-2\n            permission: CAN_QUERY\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n        - name: uc-function-1\n          uc_securable:\n            securable_full_name: ${var.catalog_name}.agent_deployment.${var.function_1_name}\n            securable_type: FUNCTION\n            permission: EXECUTE\n\ntargets:\n  dev:\n    mode: development\n    default: true'
  }
];

var e1TabCurrent = 0;

function e1Esc(s) { return s.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;'); }

function e1PyTokenize(line) {
  var out = '';
  var re = /(#.*$)|("""[\s\S]*?"""|'''[\s\S]*?'''|"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:from|import|def|class|return|if|elif|else|for|while|try|except|finally|with|as|pass|break|continue|yield|lambda|async|await|None|True|False|and|or|not|in|is)\b)|(@\w[\w.]*)|(\b\w+(?=\s*\())|(\b\d+\.?\d*\b)/g;
  var last = 0; var m;
  while ((m = re.exec(line)) !== null) {
    if (m.index > last) out += e1Esc(line.substring(last, m.index));
    if (m[1]) out += '<span class="tk-cmt">' + e1Esc(m[1]) + '</span>';
    else if (m[2]) out += '<span class="tk-str">' + e1Esc(m[2]) + '</span>';
    else if (m[3]) out += '<span class="tk-kw">' + e1Esc(m[3]) + '</span>';
    else if (m[4]) out += '<span class="tk-fn">' + e1Esc(m[4]) + '</span>';
    else if (m[5]) out += '<span class="tk-fn">' + e1Esc(m[5]) + '</span>';
    else if (m[6]) out += '<span class="tk-num">' + e1Esc(m[6]) + '</span>';
    last = m.index + m[0].length;
  }
  if (last < line.length) out += e1Esc(line.substring(last));
  return out || '\u00a0';
}

function e1YamlTokenize(line) {
  var out = '';
  var re = /(#.*$)|("(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:true|false|null)\b)|(^\s*)([\w][\w\-\.\[\]]*)(\s*[:=])/gm;
  var last = 0; var m;
  while ((m = re.exec(line)) !== null) {
    if (m.index > last) out += e1Esc(line.substring(last, m.index));
    if (m[1]) out += '<span class="tk-cmt">' + e1Esc(m[1]) + '</span>';
    else if (m[2]) out += '<span class="tk-str">' + e1Esc(m[2]) + '</span>';
    else if (m[3]) out += '<span class="tk-kw">' + e1Esc(m[3]) + '</span>';
    else if (m[5]) { out += e1Esc(m[4]) + '<span class="tk-key">' + e1Esc(m[5]) + '</span>' + e1Esc(m[6]); }
    last = m.index + m[0].length;
  }
  if (last < line.length) out += e1Esc(line.substring(last));
  return out || '\u00a0';
}

function e1Tokenize(code, lang) {
  var lines = code.split('\n');
  var result = [];
  for (var i = 0; i < lines.length; i++) {
    var raw = lines[i];
    var indent = 0;
    while (indent < raw.length && raw[indent] === ' ') indent++;
    var sp = '';
    for (var j = 0; j < indent; j++) sp += '\u00a0';
    var content = raw.substring(indent);
    var html;
    if (lang === 'python') html = e1PyTokenize(content);
    else if (lang === 'yaml' || lang === 'toml') html = e1YamlTokenize(content);
    else html = e1Esc(content) || '\u00a0';
    result.push('<span class="line">' + sp + html + '</span>');
  }
  return result.join('');
}

function e1TabSelect(id) {
  e1TabCurrent = id;
  document.querySelectorAll('.e1-tab').forEach(function(t) {
    t.classList.toggle('e1-active', parseInt(t.getAttribute('data-e1tab')) === id);
  });
  var f = E1_FILES[id];
  var btnId = 'e1-copy-btn-' + id;
  var wrapId = 'e1-code-wrap';
  var html = f.desc;
  if (f.changes) {
    html += '<div class="e1-changed-box">' + f.changes + '</div>';
  }
  html += '<div class="e1-code-wrap" id="' + wrapId + '">' +
      '<button class="e1-copy-btn" id="' + btnId + '" onclick="e1CopyCode(' + id + ')">Copy</button>' +
      '<div class="e1-code">' + e1Tokenize(f.code, f.lang) + '</div>' +
    '</div>';
  document.getElementById('e1-panel').innerHTML = html;
  document.getElementById('e1-panel-wrap').classList.add('e1-open');
}

function e1CopyCode(id) {
  var code = E1_FILES[id].code;
  var btn = document.getElementById('e1-copy-btn-' + id);
  navigator.clipboard.writeText(code).then(function() {
    btn.textContent = 'Copied!';
    setTimeout(function() { btn.textContent = 'Copy'; }, 2000);
  }, function() {
    var t = document.createElement('textarea');
    t.value = code; document.body.appendChild(t); t.select();
    document.execCommand('copy'); document.body.removeChild(t);
    btn.textContent = 'Copied!';
    setTimeout(function() { btn.textContent = 'Copy'; }, 2000);
  });
}

e1TabSelect(0);
</script>

### E2. Redeploy

Redeploy using whichever method you used in Sections C2 and D2. This update adds a UC function resource so the agent can call tools via MCP. Select a card below.

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">

<style>
.e2d-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }

.e2d-box {
  display: inline-block;
  width: 48%;
  margin-right: 4%;
  min-height: 90px;
  background: #F9F7F4;
  border-top: 8px solid transparent;
  border-left: 2px solid transparent;
  border-right: 2px solid transparent;
  border-bottom: 2px solid transparent;
  border-radius: 8px;
  padding: 12px 8px;
  text-align: center;
  box-sizing: border-box;
  cursor: pointer;
  user-select: none;
  vertical-align: top;
  transition: transform 0.12s;
}

.e2d-box:last-child { margin-right: 0; }

.e2d-box:hover { transform: translateY(-2px); }

.e2d-box.active {
  background: #fff;
  border-left-color: var(--pc);
  border-right-color: var(--pc);
  border-bottom-color: var(--pc);
}

.e2d-title {
  display: block;
  font-size: 13pt;
  font-weight: 700;
  color: #0b2026;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
}

.e2d-subtitle {
  display: block;
  font-size: 11pt;
  font-weight: 400;
  color: #666;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
  margin-top: 4px;
}

.e2d-detail-wrap {
  overflow: hidden;
  max-height: 0;
  opacity: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
  margin-top: 0;
}

.e2d-detail-wrap.open {
  max-height: 1600px;
  opacity: 1;
  margin-top: 12px;
}

.e2d-detail-card {
  background: #F9F7F4;
  border-radius: 10px;
  padding: 20px;
  box-sizing: border-box;
  border-top: 7px solid #ccc;
}

.e2d-steps-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 14px 14px;
  margin-bottom: 14px;
  font-size: 14.5pt;
  line-height: 1.6;
}

.e2d-steps-block ol {
  margin: 8px 0 0 0;
  padding-left: 20px;
}

.e2d-steps-block li {
  margin-bottom: 8px;
}

.e2d-steps-block ul {
  margin: 4px 0 8px 20px;
  list-style: disc;
}

.e2d-info-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 12px 14px;
  font-size: 14pt;
  line-height: 1.5;
  font-weight: 500;
}
</style>

<div style="background: #F8F9FC; border: 3px solid #02A36F; border-radius: 10px; padding: 24px; box-sizing: border-box;">

  <div class="e2d-row">
    <span class="e2d-box active" data-id="0" onclick="e2dSelect(0)" style="--pc:#02A36F; border-top-color:#02A36F;">
      <span class="e2d-title">DABs UI</span>
      <span class="e2d-subtitle">Git folder</span>
    </span><span class="e2d-box" data-id="1" onclick="e2dSelect(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
      <span class="e2d-title">Apps UI</span>
      <span class="e2d-subtitle">No Git folder</span>
    </span>
  </div>

  <div class="e2d-detail-wrap" id="e2d-detail-wrap">
    <div class="e2d-detail-card" id="e2d-detail-card"></div>
  </div>

</div>

</div>

<script>
var E2D_CARDS = [
  {
    title: "DABs UI (Git folder)",
    color: "#02A36F",
    bg: "rgba(2,163,111,0.10)",
    infoBg: "rgba(2,163,111,0.10)",
    content: `
      <ol>
        <li>Click the rocket icon on any file in the project folder.</li>
        <li>Click <strong>Deploy</strong>, wait for validation, then click <strong>Deploy</strong> again.</li>
        <li>Navigate to the app and click <strong>Deploy</strong> to push the updated source.</li>
        <li>Notice the new <strong>UC function</strong> resource now appears under <strong>App resources</strong>.</li>
        <li>After deployment, click on the URL to open the app.</li>
      </ol>
    `,
    info: '<strong>Requires a Git folder.</strong> The DABs UI detects the new UC function resource in <code>databricks.yml</code> and grants <code>EXECUTE</code> permission automatically. Wait for the app status to return to <strong>Running</strong>.'
  },
  {
    title: "Apps UI (no Git folder)",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    infoBg: "rgba(255,95,70,0.10)",
    content: `
      <ol>
        <li>Navigate to the app Overview page and click <strong>(⋮)</strong> icon then on  <strong>Edit</strong> at the top right.</li>
        <li>Click <strong>resources</strong> at the top of the setup wizard.</li>
        <li>Click <strong>+ Add resource</strong> and select <strong>UC function</strong>.</li>
        <li>Enter the full function name: <code>{catalog}.agent_deployment.avg_neigh_price</code> (replace <code>{catalog}</code> with your catalog name).</li>
        <li> Click <strong>Confirm</strong>.</li>
        <li>Make sure <strong>Permission</strong> is set to <strong>Can execute</strong> and <strong>Resource key</strong> is set to <strong>uc-function-1</strong>.</li>
        <li>Click <strong>Save</strong>.</li>
        <li>Back on the Overview page, click <strong>Deploy</strong> to push the updated source.</li>
      </ol>
    `,
    info: '<strong>No Git folder required.</strong> You add the UC function resource manually on the Overview page. Wait for the app status to return to <strong>Running</strong>.'
  }
];

var e2dCurrent = null;

function e2dSelect(id) {
  var wrap = document.getElementById('e2d-detail-wrap');
  var card = document.getElementById('e2d-detail-card');
  var c = E2D_CARDS[id];

  document.querySelectorAll('.e2d-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (e2dCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.e2d-box').forEach(function(b) {
      b.classList.remove('active');
    });
    e2dCurrent = null;
    return;
  }

  e2dCurrent = id;

  card.style.borderTopColor = c.color;
  card.innerHTML =
    '<div style="font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;">' + c.title + '</div>'
    + '<div class="e2d-steps-block" style="background:' + c.bg + ';border-color:' + c.color + ';">'
    + c.content
    + '</div>'
    + '<div class="e2d-info-block" style="background:' + c.infoBg + ';border-color:' + c.color + ';">'
    + c.info
    + '</div>';

  wrap.classList.add('open');
}

e2dSelect(0);
</script>

### E3. Test and Verify Tool Call Traces

The agent now has access to the `avg_neigh_price` UC function through MCP. Verify that tool calls appear in the traces.

1. Open the app URL and send a query that triggers the tool, such as: *"What is the average price for listings in the Mission neighborhood?"*
2. Wait for the agent to respond. The response should include actual data from the UC function.
3. Navigate to the **Apps** page in the workspace sidebar and select your app.
4. Under the **App resources** section, find the **MLflow Experiment ID** and click on it to open the experiment.

5. Click the **Traces** tab and open the most recent trace. You should now see:
   - The **root span** (`tool_calling_chatbot_run`) capturing the full agent invocation
   - **Child spans** from the Agents SDK runner showing each reasoning step
   - **MCP tool spans** for `tools/list` (capability discovery) and `tools/call` (the actual function invocation). Click a `tools/call` span to inspect the arguments passed to `avg_neigh_price` and the value it returned.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Compare this trace to the one from Section D3. The addition of MCP tools added new child spans without any changes to the tracing code itself. The <code>@mlflow.trace</code> decorator and the Agents SDK automatically capture MCP interactions as part of the trace tree.</p>
</div>
</div>
</div>

## F. Verify Traces in MLflow UI

Once the app is running, send a few messages through the chat interface to generate traces. Try a message that triggers a tool call, such as: *"What is the average price for listings in the Mission neighborhood?"*

1. In your Databricks workspace, navigate to **Experiments** and open the experiment named **simple-chatbot-experiment** (under your user path).
2. Click the **Traces** tab. You should see a trace entry for each message you sent.
3. Click a trace to expand it. You will see:
   - The **root span** (`tool_calling_chatbot_run`) capturing the full agent invocation: inputs, final output, and total latency
   - **Child spans** from the Agents SDK runner showing each step: the initial completion, tool call decision, MCP round-trip, and final synthesis
   - **MCP tool spans** showing the arguments sent to the UC function and the value it returned
4. Back on the experiment landing page, click the **Evaluation runs** tab. Each invocation appears as a nested run with logged parameters (`agent_name`, `model_endpoint`, `num_messages`) and artifacts (`inputs/messages.json`, `outputs/final_output.txt`).

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Traces vs. Runs</strong>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Traces</strong> capture the execution graph of a single request (spans, timing, inputs, and outputs) and are best for debugging individual interactions.</li>
<li><strong>Runs</strong> capture experiment-level metadata (parameters, metrics, and artifacts) and are best for comparing behavior across many interactions over time.</li>
</ul>
</div>
</div>
</div>

## G. Clean Up

When you are finished with this demo, remove the resources created during deployment.

1. Navigate to **Apps** under **Compute** and select the three vertical dots next to your app. Select **Delete** to stop and remove the app.
2. Navigate to **Experiments**, open the experiment, and delete it if you no longer need the trace and run history.

## H. Conclusion

In this demo, you progressively built a production-ready agent deployed as a Databricks App:

1. **Deployed a basic agent** using the OpenAI Agents SDK, FastAPI, and Declarative Automation Bundles
2. **Added MLflow tracing** with `@mlflow.trace` and `mlflow.start_run` to capture per-request execution spans, parameters, and artifacts
3. **Integrated MCP tools** by connecting a Unity Catalog function through a Managed MCP server, making real data lookups available to the agent
4. **Verified traces** in the MLflow UI, inspecting root spans, Agents SDK child spans, and MCP tool call spans

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

